<a href="https://colab.research.google.com/github/shijaalive-cloud/data_-_rag-/blob/main/data%26rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-community

In [ ]:
import pandas as pd

# Load the Excel file into a DataFrame
df = pd.read_excel('/content/NCT_Nigeria.xlsx')

# Display the first 5 rows
display(df.head())

In [ ]:
from langchain_core.documents import Document


In [ ]:
import pandas as pd
from langchain_core.documents import Document

# Ensure df is defined by loading the Excel dataset
try:
    df
except NameError:
    df = pd.read_excel('/content/NCT_Nigeria.xlsx')

documents = []

for _, row in df.iterrows():
    # Skip rows that don't have a valid Code or Food Description (handles NaN rows)
    if pd.isna(row['Code']) or pd.isna(row['Food item description in English']):
        continue

    # Construct a comprehensive prose summary of the row data
    content = (
        f"Food Item: {row['Food item description in English']}. "
        f"It yields {row['Energy         (kcal)']} kcal of energy per portion. "
        f"Nutritional breakdown: Protein: {row['Protein, total (g) ']}g, "
        f"Fat: {row['Fat                     (g) ']}g, "
        f"Carbohydrates: {row['Carbohydrate, available, calculated by difference      (g)']}g, "
        f"Fiber: {row['Total Fiber      (g) ']}g, "
        f"Calcium: {row['Calcium       (mg)']}mg, "
        f"Iron: {row['Iron               (mg)']}mg, "
        f"Potassium: {row['Potassium (mg)']}mg, "
        f"Vitamin C: {row['Vitamin C     (mg)']}mg."
    )

    # Store clean textual context alongside raw mapping metadata
    doc = Document(
        page_content=content,
        metadata={"code": int(row['Code']) if isinstance(row['Code'], (int, float)) else row['Code'], "name": row['Food item description in English']}
    )
    documents.append(doc)

print(f"Loaded {len(documents)} structured food items into document chunks.")

In [ ]:
from langchain_core.documents import Document
import pandas as pd

documents = []

for _, row in df.iterrows():
    # Skip rows that don't have a valid Code or Food Description (handles NaN rows)
    if pd.isna(row['Code']) or pd.isna(row['Food item description in English']):
        continue

    # Construct a comprehensive prose summary of the row data
    content = (
        f"Food Item: {row['Food item description in English']}. "
        f"It yields {row['Energy         (kcal)']} kcal of energy per portion. "
        f"Nutritional breakdown: Protein: {row['Protein, total (g) ']}g, "
        f"Fat: {row['Fat                     (g) ']}g, "
        f"Carbohydrates: {row['Carbohydrate, available, calculated by difference      (g)']}g, "
        f"Fiber: {row['Total Fiber      (g) ']}g, "
        f"Calcium: {row['Calcium       (mg)']}mg, "
        f"Iron: {row['Iron               (mg)']}mg, "
        f"Potassium: {row['Potassium (mg)']}mg, "
        f"Vitamin C: {row['Vitamin C     (mg)']}mg."
    )

    # Store clean textual context alongside raw mapping metadata
    doc = Document(
        page_content=content,
        metadata={"code": row['Code'], "name": row['Food item description in English']}
    )
    documents.append(doc)

print(f"Loaded {len(documents)} structured food items into document chunks.")

In [ ]:
# Install required integration packages
!pip install -q langchain-openai faiss-cpu

import os
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Securely fetch the API key from Colab Secrets
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except Exception:
    print("Warning: 'OPENAI_API_KEY' secret not found in Colab Secrets.")
    print("Please add your API key in the Secrets tab (key icon on the left) named 'OPENAI_API_KEY'.")

# 1. Initialize OpenAI's high-performance compact model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# 2. Build the FAISS similarity index from your food documents list
vector_store = FAISS.from_documents(documents, embedding_model)

# 3. Create the retriever function
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.3 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)
